# 단일 클래스 알약 탐지기 학습

`data/`의 고정 분할을 읽어 `train_loader`, `val_loader`, `label_map`, `data_summary`를 구성한 뒤 SSDLite 학습·재개·평가·artifact 저장을 실행한다. 모델, epoch loop, 지표와 checkpoint 로직은 `detector/` 모듈을 사용한다.

In [ ]:
import sys
from importlib import import_module
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import pandas as pd

from configs.detector import DetectorConfig
from detector import (
    evaluate_model,
    load_inference_model,
    plot_detection_comparison,
    predict_detector,
    save_inference_artifacts,
)
from detector.train import train_detector
from detector.training_data import prepare_detector_data
from pipeline.pill_recognition import build_classifier_batch

In [ ]:
run_name = "ssdlite320_baseline"
batch_size = 8
max_epochs = 50
resume_from = None  # 재개 예: output_root / run_name / "last.pt"
pretrained_backbone = True
data_factory_path = "pilldata.detector_dataset:build_detector_inputs"
data_settings_path = None  # 외부 데이터 예: Path("/content/drive/MyDrive/detector_data_settings.json")
output_root = project_root / "outputs"
# Colab에서 결과를 Drive에 저장하려면 output_root를 Drive 경로로 변경하세요.

run_dir = output_root / run_name

config = DetectorConfig(
    run_name=run_name,
    output_dir=output_root,
    label_map_path=run_dir / "label_map.json",
    data_summary_path=run_dir / "data_summary.json",
    learning_rate=5e-3,
    weight_decay=5e-4,
    score_threshold=0.25,
    nms_threshold=0.45,
    early_stopping_patience=7,
    data_factory_path=data_factory_path,
    data_settings_path=data_settings_path,
    pretrained_backbone=pretrained_backbone,
    device="auto",
    batch_size=batch_size,
    max_epochs=max_epochs,
    resume_from=resume_from,
    show_progress=True,
)
print(
    f"device={config.device}, model={config.model_name}, "
    f"batch_size={config.batch_size}, max_epochs={config.max_epochs}, "
    f"resume_from={config.resume_from}"
)


## 데이터 연결

`data_factory_path`가 가리키는 함수는 `factory(config=config, device=device)` 호출을 지원하고 `(train_loader, val_loader, label_map, data_summary)`를 반환해야 한다. 정확한 형식은 `docs/detector_input_spec.md`를 따른다.

In [ ]:
if not data_factory_path:
    raise RuntimeError("data_factory_path를 데이터 구성 모듈의 함수로 설정하세요")

training_data = prepare_detector_data(config)

In [ ]:
train_loader = training_data.train_loader
val_loader = training_data.val_loader
label_map = training_data.label_map
data_summary = training_data.data_summary
device = training_data.device
print(f"입력 검증 완료: device={device}")

## 학습 전 bbox 확인

입력 검증은 원본 annotation 자체에서 빠진 알약을 찾을 수 없다. 학습 전에 여러 무작위 샘플에서 이미지 속 모든 알약에 bbox가 있는지 확인한다.

In [ ]:
preview_batch = next(iter(val_loader))
preview_figure = plot_detection_comparison(
    preview_batch["images"][0],
    [],
    target=preview_batch["targets"][0],
    title=f"GT image_id={preview_batch['meta'][0]['image_id']}",
)
preview_figure

## 신규 학습 또는 재개

재개할 때는 `config.resume_from`을 같은 실행의 `last.pt`로 설정한다. 첫 실행에서는 ImageNet backbone 가중치를 내려받을 수 있다.

In [ ]:
# config.resume_from = output_root / run_name / "last.pt"
training_result = train_detector(
    config,
    data=training_data,
)
training_result

In [ ]:
metrics = pd.read_csv(training_result.run_dir / "metrics.csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
metrics.plot(x="epoch", y="train_loss", ax=axes[0, 0], title="Training loss")
metrics.plot(
    x="epoch",
    y=["map", "map_50", "map_75", "map_75_95"],
    ax=axes[0, 1],
    title="Validation average precision",
)
metrics.plot(
    x="epoch",
    y=["mean_average_recall", "recall_before_threshold", "recall_after_threshold"],
    ax=axes[1, 0],
    title="Validation recall",
)
metrics.plot(
    x="epoch",
    y=["mean_count_error_before_threshold", "mean_count_error_after_threshold"],
    ax=axes[1, 1],
    title="Prediction count error",
)
fig.tight_layout()

## Best checkpoint 평가와 예측 저장

In [ ]:
best_model, device = load_inference_model(
    training_result.best_checkpoint_path,
    label_map,
    device=device,
)
best_metrics = evaluate_model(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    show_progress=config.show_progress,
    progress_description="Detector final validation",
)
inference_result = predict_detector(
    best_model,
    val_loader,
    device,
    score_threshold=config.score_threshold,
    detector_name=config.model_name,
)
save_inference_artifacts(inference_result, training_result.run_dir)
best_metrics.to_dict()

In [ ]:
first_image_id = preview_batch["meta"][0]["image_id"]
first_predictions = [
    prediction
    for prediction in inference_result.predictions
    if prediction.image_id == first_image_id
]
comparison_figure = plot_detection_comparison(
    preview_batch["images"][0],
    first_predictions,
    target=preview_batch["targets"][0],
    title=f"GT vs prediction image_id={first_image_id}",
)
visualization_dir = training_result.run_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
comparison_figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
comparison_figure

## 선택: 분류기 연결 batch 확인

아래 셀은 crop batch만 만든다. 분류기 모델과 class map은 분류기 실행 코드에서 직접 전달한다.

In [ ]:
classifier_batch = build_classifier_batch(
    preview_batch["images"][0],
    first_predictions,
)
print(
    f"classifier images={tuple(classifier_batch['images'].shape)}, "
    f"detections={len(classifier_batch['meta'])}"
)